# Hinglish Support Intent — QLoRA fine-tune of Qwen2.5-0.5B

Runs end to end on a **free Colab T4** in about 20–30 minutes.

**Runtime → Change runtime type → T4 GPU**, then *Runtime → Run all*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!git clone https://github.com/Abhisheklpu1731/hinglish-intent-qlora.git || git -C hinglish-intent-qlora pull
%cd /content/hinglish-intent-qlora
!pip install -q -r requirements.txt

## 1. Generate the synthetic dataset
Test-set sentence patterns are held out from training, so the test measures generalisation to new phrasings, not memorised templates.

In [ ]:
!python data/generate.py --out data
!head -3 data/test.jsonl

## 2. Baseline: zero-shot base model

In [ ]:
!python src/evaluate.py --model Qwen/Qwen2.5-0.5B-Instruct --tag base

## 3. LoRA fine-tune (r=16 on all linear layers)
`--fp32` trains in full precision, which runs reliably on the free T4. On an Ampere GPU (A100, L4), drop `--fp32` to train with QLoRA (4-bit NF4 base, bf16).

In [ ]:
!python src/train.py --model Qwen/Qwen2.5-0.5B-Instruct --out outputs/qwen05-lora --epochs 2 --fp32

## 4. Evaluate the fine-tuned model

In [ ]:
!python src/evaluate.py --model Qwen/Qwen2.5-0.5B-Instruct --adapter outputs/qwen05-lora --tag finetuned

## 5. Results

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
rows = {t: json.load(open(f"results/{t}_metrics.json")) for t in ["base", "finetuned"]}
cols = ["json_valid", "intent_accuracy", "intent_macro_f1", "order_id_exact", "ms_per_example"]
df = pd.DataFrame(rows).T[cols]
display(df)
print(df.to_markdown())   # paste this table into README.md

log = json.load(open("outputs/qwen05-lora/train_log.json"))
tr = [(h["step"], h["loss"]) for h in log if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in log if "eval_loss" in h]
plt.plot(*zip(*tr), label="train"); plt.plot(*zip(*ev), "o-", label="eval")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("SFT loss")
plt.savefig("results/loss.png", dpi=120, bbox_inches="tight"); plt.show()

## 6. Error analysis — what does the fine-tuned model still get wrong?

In [ ]:
from collections import Counter
import sys; sys.path.insert(0, "src")
from common import parse_output
preds = [json.loads(l) for l in open("results/finetuned_predictions.jsonl")]
wrong = [p for p in preds if parse_output(p["raw_output"])[0] != p["intent"]]
print(f"{len(wrong)} / {len(preds)} intent errors")
print(Counter((p["intent"], parse_output(p["raw_output"])[0]) for p in wrong).most_common(10))
pd.DataFrame(wrong)[["text", "intent", "raw_output"]].head(15)

## 7. Save results and push the adapter to Hugging Face
The adapter is a few MB. Pushing it lets the free Hugging Face Space demo load it (see `space/`).
Create a **write** token at huggingface.co/settings/tokens when the login box asks.

In [ ]:
!zip -rq results.zip results outputs/qwen05-lora/train_log.json
from google.colab import files; files.download("results.zip")   # commit results/ to the repo

from huggingface_hub import notebook_login, HfApi
notebook_login()
api = HfApi()
repo_id = f"{api.whoami()['name']}/qwen05-hinglish-intent-lora"
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(folder_path="outputs/qwen05-lora", repo_id=repo_id,
                  ignore_patterns=["checkpoint-*", "*.pt", "train_log.json"])
print("Adapter pushed:", f"https://huggingface.co/{repo_id}")